# Data extration from .tar file

# First visit WGAN-GP. I explain that in detail.Here I skip so

In [ ]:
import tarfile
import os

source_path = "/kaggle/input/datasets/dschettler8845/brats-2021-task1/BraTS2021_Training_Data.tar"
extract_path = "/kaggle/working/brats2021"

os.makedirs(extract_path, exist_ok=True)

with tarfile.open(source_path) as tar:
    tar.extractall(path=extract_path)

print("Extraction complete!")


# Libraries

In [ ]:
# -------------------------------
# Libraries
# -------------------------------
import numpy as np
import nibabel as nib
import cv2
import os
import tensorflow as tf
from tensorflow.keras import layers
from tqdm import tqdm
import matplotlib.pyplot as plt
from IPython.display import clear_output
import time


# Same way as WGAN-GP images process

In [ ]:
# -------------------------------
# Preprocessing
# -------------------------------
IMG_SIZE = 128
extract_path = "/kaggle/working/brats2021"

def normalize(img):
    img = img.astype(np.float32)
    mean = np.mean(img)
    std = np.std(img)
    if std == 0:
        return img
    img = (img - mean) / std
    img = (img - img.min()) / (img.max() - img.min() + 1e-8)
    img = img * 2.0 - 1.0
    return img.astype("float16")

tumor_images = []

patient_folders = sorted([
    f for f in os.listdir(extract_path)
    if os.path.isdir(os.path.join(extract_path, f)) and "BraTS" in f
])

print("Total patients:", len(patient_folders))

for patient in tqdm(patient_folders):
    patient_path = os.path.join(extract_path, patient)
    files = os.listdir(patient_path)

    flair_path = [f for f in files if "flair" in f][0]
    seg_path   = [f for f in files if "seg" in f][0]

    flair = nib.load(os.path.join(patient_path, flair_path)).get_fdata()
    seg   = nib.load(os.path.join(patient_path, seg_path)).get_fdata()

    start = int(flair.shape[2] * 0.2)
    end   = int(flair.shape[2] * 0.8)

    for i in range(start, end):
        tumor_pixels = np.sum(seg[:, :, i] > 0)
        brain_pixels = np.sum(flair[:, :, i] > 0)

        if tumor_pixels > 200 and brain_pixels > 15000: # for >15000 filter images 41k and >12000 images 57k 
            img_slice  = flair[:, :, i]
            coords = np.argwhere(img_slice > np.percentile(img_slice, 5))
            if coords.shape[0] == 0:
                continue

            y_min, x_min = coords.min(axis=0)
            y_max, x_max = coords.max(axis=0)

            pad = 10
            y_min = max(0, y_min - pad)
            x_min = max(0, x_min - pad)
            y_max = min(img_slice.shape[0], y_max + pad)
            x_max = min(img_slice.shape[1], x_max + pad)

            cropped_img  = img_slice[y_min:y_max, x_min:x_max]

            img  = cv2.resize(cropped_img,  (IMG_SIZE, IMG_SIZE))
            img  = normalize(img)
            img  = np.expand_dims(img, axis=-1)

            tumor_images.append(img)

tumor_images = np.array(tumor_images, dtype="float16")
print("Filtered Dataset Shape:", tumor_images.shape)


# Total paient 1251 and there 3D images

# Some images after Preprocess

In [ ]:
# last >12000 filter
import matplotlib.pyplot as plt
import numpy as np

num_images = 20

indices = np.random.choice(len(tumor_images), num_images, replace=False)

plt.figure(figsize=(10, 8))

for i, idx in enumerate(indices):
    plt.subplot(4, 5, i + 1)
    
    img = tumor_images[idx].squeeze()
    
    # convert from [-1,1] → [0,1]
    img = (img + 1) / 2
    
    plt.imshow(img, cmap='gray')
    plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# new
import matplotlib.pyplot as plt
import numpy as np

num_images = 20

indices = np.random.choice(len(tumor_images), num_images, replace=False)

plt.figure(figsize=(10, 8))

for i, idx in enumerate(indices):
    plt.subplot(4, 5, i + 1)
    
    img = tumor_images[idx].squeeze()
    
    # convert from [-1,1] → [0,1]
    img = (img + 1) / 2
    
    plt.imshow(img, cmap='gray')
    plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

num_images = 20

indices = np.random.choice(len(tumor_images), num_images, replace=False)

plt.figure(figsize=(10, 8))

for i, idx in enumerate(indices):
    plt.subplot(4, 5, i + 1)
    
    img = tumor_images[idx].squeeze()
    
    # convert from [-1,1] → [0,1]
    img = (img + 1) / 2
    
    plt.imshow(img, cmap='gray')
    plt.axis('off')

plt.tight_layout()
plt.show()

# Hyperparameters. I also add early stopping for inital train time. But for this one it is not needed.I give epoch after modification

In [ ]:
# -------------------------------
# GAN Hyperparameters
# -------------------------------
BATCH_SIZE = 32
NOISE_DIM = 100
EPOCHS = 15       # recommended for 128x128
IMG_SIZE = 128
PATIENCE = 15       # early stopping patience


# Shuffle the dataset for randomness

In [ ]:
# -------------------------------
# Dataset for GAN
# -------------------------------
def create_dataset(images, target_size=IMG_SIZE):
    def resize(img):
        img = tf.image.resize(img, (target_size, target_size))
        return img
    ds = tf.data.Dataset.from_tensor_slices(images)
    ds = ds.shuffle(5000)
    ds = ds.map(resize, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds

dataset = create_dataset(tumor_images, target_size=IMG_SIZE)

# Generator same as WGAN-GP

# Dense layer 512->256->128->64->32 .there is extran one layer than WGAN-GP

In [ ]:
# -------------------------------
# Generator
# -------------------------------
def build_generator():
    noise = layers.Input(shape=(NOISE_DIM,))
    x = layers.Dense(8*8*512, use_bias=False)(noise)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU()(x)
    x = layers.Reshape((8,8,512))(x)

    x = layers.Conv2DTranspose(256, 4, 2, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU()(x)

    x = layers.Conv2DTranspose(128, 4, 2, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU()(x)

    x = layers.Conv2DTranspose(64, 4, 2, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU()(x)
    x = layers.Conv2DTranspose(32, 4, 2, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU()(x)

    output = layers.Conv2DTranspose(1, 3, 1, padding='same', activation='tanh')(x)
    return tf.keras.Model(noise, output)

generator = build_generator()

# Discriminator 
* Conv2D
* LeakyReLU
* Dense layer 32->64->32->128->256->512

In [ ]:
# -------------------------------
# Discriminator
# -------------------------------
def build_discriminator():
    img = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 1))
    x = layers.Conv2D(32, 4, 2, padding='same')(img)
    x = layers.LeakyReLU()(x)

    x = layers.Conv2D(64, 4, 2, padding='same')(x)
    x = layers.LeakyReLU()(x)

    x = layers.Conv2D(128, 4, 2, padding='same')(x)
    x = layers.LeakyReLU()(x)

    x = layers.Conv2D(256, 4, 2, padding='same')(x)
    x = layers.LeakyReLU()(x)

    x = layers.Conv2D(512, 4, 2, padding='same')(x)
    x = layers.LeakyReLU()(x)

    x = layers.Flatten()(x)
    output = layers.Dense(1)(x)
    return tf.keras.Model(img, output)
discriminator = build_discriminator()


* loss is BinaryCrossentropy
* add label smothing for real label (*0.9)
* learning rate G=1e-4, D=2e-4 and beta=0.5 ,0.999

In [ ]:
# -------------------------------
# Loss & Optimizer
# -------------------------------
cross_entropy = tf.keras.losses.BinaryCrossentropy(from_logits=True)

def generator_loss(fake_output):
    return cross_entropy(tf.ones_like(fake_output), fake_output)

def discriminator_loss(real_output, fake_output):
    real_labels = tf.ones_like(real_output) * 0.9
    fake_labels = tf.zeros_like(fake_output)
    return cross_entropy(real_labels, real_output) + cross_entropy(fake_labels, fake_output)

generator_optimizer = tf.keras.optimizers.Adam(1e-4, beta_1=0.5, beta_2=0.999)
discriminator_optimizer = tf.keras.optimizers.Adam(2e-4, beta_1=0.5, beta_2=0.999)


# Training step

In [ ]:
# -------------------------------
# Training Step
# -------------------------------
@tf.function
def train_step(images):
    batch_size = tf.shape(images)[0]
    noise = tf.random.normal([batch_size, NOISE_DIM])
    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
        generated_images = generator(noise, training=True)
        real_output = discriminator(images, training=True)
        fake_output = discriminator(generated_images, training=True)
        gen_loss = generator_loss(fake_output)
        disc_loss = discriminator_loss(real_output, fake_output)

    generator_optimizer.apply_gradients(zip(gen_tape.gradient(gen_loss, generator.trainable_variables),
                                            generator.trainable_variables))
    discriminator_optimizer.apply_gradients(zip(disc_tape.gradient(disc_loss, discriminator.trainable_variables),
                                                discriminator.trainable_variables))
    return gen_loss, disc_loss

# Plot function to see generated images in each epoch

In [ ]:
# -------------------------------
# Plot Function
# -------------------------------
def generate_and_plot(epoch):
    noise = tf.random.normal([4, NOISE_DIM])
    predictions = generator(noise, training=False)
    predictions = (predictions + 1)/2
    plt.figure(figsize=(7,6))
    for i in range(4):
        plt.subplot(1,4,i+1)
        plt.imshow(predictions[i,:,:,0], cmap='gray')
        plt.axis('off')
    plt.suptitle(f"Epoch {epoch}", fontsize=20)
    plt.tight_layout()
    plt.show()

# Training loop
# epoch time = 184.14 Sec

In [ ]:
# -------------------------------
# Training Loop with Checkpoints and Early Stopping
# -------------------------------
checkpoint_dir = './checkpoints_dcgan'
os.makedirs(checkpoint_dir, exist_ok=True)
checkpoint_prefix = os.path.join(checkpoint_dir, "ckpt")
best_gen_loss = np.inf
wait = 0

for epoch in range(EPOCHS):
    start = time.time()
    progbar = tqdm(dataset, desc=f"Epoch {epoch+1}/{EPOCHS}")
    epoch_gen_loss = []

    for image_batch in progbar:
        gen_loss, disc_loss = train_step(image_batch)
        epoch_gen_loss.append(gen_loss.numpy())
        progbar.set_postfix({
            "G": f"{gen_loss.numpy():.3f}",
            "D": f"{disc_loss.numpy():.3f}"
        })
    clear_output(wait=True)
    avg_gen_loss = np.mean(epoch_gen_loss)
    print(f"Epoch {epoch+1} | Gen Loss: {avg_gen_loss:.4f} | Time: {time.time()-start:.2f}s")
    # -------------------------------
    # Save Checkpoint every 10 epochs
    # -------------------------------
    if (epoch+1) % 10 == 0:
        generator.save(os.path.join(checkpoint_dir, f'generator_epoch_{epoch+1}.h5'))
        discriminator.save(os.path.join(checkpoint_dir, f'discriminator_epoch_{epoch+1}.h5'))

    # -------------------------------
    # Early Stopping
    # -------------------------------
    if avg_gen_loss < best_gen_loss:
        best_gen_loss = avg_gen_loss
        wait = 0
    else:
        wait += 1
        if wait >= PATIENCE:
            print(f"Early stopping at epoch {epoch+1} (no improvement in {PATIENCE} epochs)")
            break

    # Plot generated images every 5 epochs
    generate_and_plot(epoch+1)
   

# Generated Fake images

In [ ]:
# -------------------------------
# Safe Batch Generation
# -------------------------------
NUM_GEN = 3000
GEN_BATCH = 32   # VERY IMPORTANT (16 or 32 safe)

generated_images = []

for i in range(0, NUM_GEN, GEN_BATCH):
    noise = tf.random.normal([GEN_BATCH, NOISE_DIM])
    fake_batch = generator(noise, training=False)

    # fake_batch = (fake_batch + 1) / 2  # scale to [0,1]
    generated_images.append(fake_batch.numpy())

generated_images = np.concatenate(generated_images, axis=0)

print("Generated shape:", generated_images.shape)

# Take 2000 Real and 2000 Fake images for FID and IS Calculation

In [ ]:
NUM = 2000

real_images = tumor_images[:NUM]
# real_images = (real_images + 1) / 2

 fake_images = generated_images[:NUM]

In [ ]:
# new
# Shuffle both
real_idx = np.random.choice(len(real_images), 8, replace=False)
fake_idx = np.random.choice(len(fake_images), 8, replace=False)

real_sample = real_images[real_idx]
fake_sample = fake_images[fake_idx]

plt.figure(figsize=(7,6))  # 🔥 bigger

# 🔵 Real images (TOP ROWS)
for i in range(8):
    plt.subplot(4,4,i+1)
    plt.imshow(real_sample[i,:,:,0], cmap='gray')
    plt.axis('off')

# 🟢 Fake images (BOTTOM ROWS)
for i in range(8):
    plt.subplot(4,4,i+9)
    plt.imshow(fake_sample[i,:,:,0], cmap='gray')
    plt.axis('off')

plt.suptitle("Top: Shuffled Real MRI | Bottom: Shuffled Generated MRI (DCGAN)", fontsize=20)
plt.tight_layout()
plt.show()

In [ ]:
# new
# Shuffle both
real_idx = np.random.choice(len(real_images), 8, replace=False)
fake_idx = np.random.choice(len(fake_images), 8, replace=False)

real_sample = real_images[real_idx]
fake_sample = fake_images[fake_idx]

plt.figure(figsize=(7,6))  # 🔥 bigger

# 🔵 Real images (TOP ROWS)
for i in range(8):
    plt.subplot(4,4,i+1)
    plt.imshow(real_sample[i,:,:,0], cmap='gray')
    plt.axis('off')

# 🟢 Fake images (BOTTOM ROWS)
for i in range(8):
    plt.subplot(4,4,i+9)
    plt.imshow(fake_sample[i,:,:,0], cmap='gray')
    plt.axis('off')

plt.suptitle("Top: Shuffled Real MRI | Bottom: Shuffled Generated MRI (DCGAN)", fontsize=20)
plt.tight_layout()
plt.show()

In [ ]:
# new
# Shuffle both
real_idx = np.random.choice(len(real_images), 8, replace=False)
fake_idx = np.random.choice(len(fake_images), 8, replace=False)

real_sample = real_images[real_idx]
fake_sample = fake_images[fake_idx]

plt.figure(figsize=(8,6))  # 🔥 bigger

# 🔵 Real images (TOP ROWS)
for i in range(8):
    plt.subplot(4,4,i+1)
    plt.imshow(real_sample[i,:,:,0], cmap='gray')
    plt.axis('off')

# 🟢 Fake images (BOTTOM ROWS)
for i in range(8):
    plt.subplot(4,4,i+9)
    plt.imshow(fake_sample[i,:,:,0], cmap='gray')
    plt.axis('off')

plt.suptitle("Top: Shuffled Real MRI | Bottom: Shuffled Generated MRI (DCGAN)", fontsize=20)
plt.tight_layout()
plt.show()

In [ ]:
np.save("generated_images_DCGAN2.npy", generated_images)


# Version1

In [ ]:
# version 1
data=np.load("/kaggle/input/datasets/shantaakterid170/generated-dcgan/generated_images_DCGAN2.npy")
print("Shape:", data.shape)
print("Min:", data.min(), "Max:", data.max())

In [ ]:
print("Shape:", real_images.shape)
print("Min:", real_images.min(), "Max:", real_images.max())

# Version 2

In [ ]:
# version2
data2=np.load("/kaggle/input/datasets/shantaakterid170/generated-dcgan/generated_images_DCGAN.npy")
print("Shape:", data2.shape)
print("Min:", data2.min(), "Max:", data2.max())

In [ ]:
NUM = 2000

real_images = tumor_images[:NUM]
real_images = (real_images + 1) / 2

In [ ]:
print("Shape:", real_images.shape)
print("Min:", real_images.min(), "Max:", real_images.max())

# Version 3

In [ ]:
# version3
data3=np.load("/kaggle/input/datasets/shantaakterid170/generated-dcgan-2/generated_images_128.npy")
print("Shape:", data3.shape)
print("Min:", data3.min(), "Max:", data3.max())

In [ ]:
fake_images3=data3[:2000]
print("Real:", real_images.shape)
print("Fake:", fake_images3.shape)

# FID Calculation Same as WGAN-GP

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications.inception_v3 import preprocess_input

def preprocess_images(images):
    # assume images are in [-1,1]
    images = (images + 1.0) * 127.5
    
    # grayscale → RGB
    images = np.repeat(images, 3, axis=-1)
    
    # resize to 299x299
    images = tf.image.resize(images, (299, 299)).numpy()
    
    return preprocess_input(images)

real_proc = preprocess_images(real_images)
fake_proc = preprocess_images(fake_images)

In [ ]:
from tensorflow.keras.applications import InceptionV3

model = InceptionV3(include_top=False, pooling='avg', input_shape=(299,299,3))

In [ ]:
real_features = model.predict(real_proc, batch_size=32, verbose=1)
fake_features = model.predict(fake_proc, batch_size=32, verbose=1)

# Version1 FID

In [ ]:
# last
from scipy import linalg

def calculate_fid(f1, f2):
    mu1, sigma1 = f1.mean(axis=0), np.cov(f1, rowvar=False)
    mu2, sigma2 = f2.mean(axis=0), np.cov(f2, rowvar=False)

    diff = mu1 - mu2
    covmean, _ = linalg.sqrtm(sigma1.dot(sigma2), disp=False)

    if np.iscomplexobj(covmean):
        covmean = covmean.real

    return diff.dot(diff) + np.trace(sigma1 + sigma2 - 2 * covmean)

fid_score = calculate_fid(real_features, fake_features)
print("🎯 FID Score:", fid_score)

In [ ]:
# data = np.load("WGANgenerated_images_128.npy")

print("Shape:", real_images.shape)
print("Min:", real_images.min(), "Max:", real_images.max())

In [ ]:
# data = np.load("WGANgenerated_images_128.npy")

print("Shape:", fake_images.shape)
print("Min:", fake_images.min(), "Max:", fake_images.max())

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications.inception_v3 import preprocess_input

def preprocess_images(images):
    """
    Preprocess images for InceptionV3 FID calculation.
    Assumes images are in [0,1].
    Converts grayscale to RGB and resizes to 299x299.
    """
    # scale [0,1] → [0,255]
    images = images * 255.0
    
    # grayscale → RGB
    if images.shape[-1] == 1:
        images = np.repeat(images, 3, axis=-1)
    
    # resize to 299x299
    images = tf.image.resize(images, (299, 299)).numpy()
    
    # preprocess for InceptionV3
    return preprocess_input(images)

# Example usage
real_proc = preprocess_images(real_images[:2000])
fake_proc = preprocess_images(fake_images[:2000])

In [ ]:
from tensorflow.keras.applications import InceptionV3

model = InceptionV3(include_top=False, pooling='avg', input_shape=(299,299,3))

In [ ]:
real_features = model.predict(real_proc, batch_size=32, verbose=1)
fake_features = model.predict(fake_proc, batch_size=32, verbose=1)

# Version2 FID

In [ ]:
from scipy import linalg

def calculate_fid(f1, f2):
    mu1, sigma1 = f1.mean(axis=0), np.cov(f1, rowvar=False)
    mu2, sigma2 = f2.mean(axis=0), np.cov(f2, rowvar=False)

    diff = mu1 - mu2
    covmean, _ = linalg.sqrtm(sigma1.dot(sigma2), disp=False)

    if np.iscomplexobj(covmean):
        covmean = covmean.real

    return diff.dot(diff) + np.trace(sigma1 + sigma2 - 2 * covmean)

fid_score = calculate_fid(real_features, fake_features)
print("🎯 FID Score:", fid_score)

# Version3

In [ ]:
from scipy import linalg

def calculate_fid(f1, f2):
    mu1, sigma1 = f1.mean(axis=0), np.cov(f1, rowvar=False)
    mu2, sigma2 = f2.mean(axis=0), np.cov(f2, rowvar=False)

    diff = mu1 - mu2
    covmean, _ = linalg.sqrtm(sigma1.dot(sigma2), disp=False)

    if np.iscomplexobj(covmean):
        covmean = covmean.real

    return diff.dot(diff) + np.trace(sigma1 + sigma2 - 2 * covmean)

fid_score = calculate_fid(real_features, fake_features)
print("🎯 FID Score:", fid_score)

# IS score same as WGAN-GP

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications.inception_v3 import preprocess_input

def preprocess_images(images):
    # [-1,1] → [0,255]
    # version1
    # images = (images + 1.0) * 127.5
    # Version2
    images = images * 255.0

    
    # grayscale → RGB
    images = np.repeat(images, 3, axis=-1)
    
    # resize
    images = tf.image.resize(images, (299, 299)).numpy()
    
    return preprocess_input(images)

fake_proc = preprocess_images(fake_images)

In [ ]:
from tensorflow.keras.applications import InceptionV3

model = InceptionV3(include_top=True, weights='imagenet')

In [ ]:
preds = model.predict(fake_proc, batch_size=32, verbose=1)

# Version1 IS

In [ ]:
# last
import numpy as np
from scipy.stats import entropy

def calculate_inception_score(preds, splits=10):
    scores = []
    N = preds.shape[0]
    
    for i in range(splits):
        part = preds[i * N // splits : (i + 1) * N // splits]
        
        py = np.mean(part, axis=0)
        kl = part * (np.log(part + 1e-10) - np.log(py + 1e-10))
        kl_sum = np.mean(np.sum(kl, axis=1))
        
        scores.append(np.exp(kl_sum))
    
    return np.mean(scores), np.std(scores)

is_mean, is_std = calculate_inception_score(preds)

print(f"🎯 Inception Score: {is_mean:.4f} ± {is_std:.4f}")

# Version2 IS

In [ ]:
# new
import numpy as np
from scipy.stats import entropy

def calculate_inception_score(preds, splits=10):
    scores = []
    N = preds.shape[0]
    
    for i in range(splits):
        part = preds[i * N // splits : (i + 1) * N // splits]
        
        py = np.mean(part, axis=0)
        kl = part * (np.log(part + 1e-10) - np.log(py + 1e-10))
        kl_sum = np.mean(np.sum(kl, axis=1))
        
        scores.append(np.exp(kl_sum))
    
    return np.mean(scores), np.std(scores)

is_mean, is_std = calculate_inception_score(preds)

print(f"🎯 Inception Score: {is_mean:.4f} ± {is_std:.4f}")

# Version3 IS

In [ ]:
# new
import numpy as np
from scipy.stats import entropy

def calculate_inception_score(preds, splits=10):
    scores = []
    N = preds.shape[0]
    
    for i in range(splits):
        part = preds[i * N // splits : (i + 1) * N // splits]
        
        py = np.mean(part, axis=0)
        kl = part * (np.log(part + 1e-10) - np.log(py + 1e-10))
        kl_sum = np.mean(np.sum(kl, axis=1))
        
        scores.append(np.exp(kl_sum))
    
    return np.mean(scores), np.std(scores)

is_mean, is_std = calculate_inception_score(preds)

print(f"🎯 Inception Score: {is_mean:.4f} ± {is_std:.4f}")

# Generated Images

# Version 1

In [ ]:
# new
import matplotlib.pyplot as plt
import numpy as np

# Select 8 random images
idx_real = np.random.choice(len(real_images), 8, replace=False)
idx_fake = np.random.choice(len(fake_images), 8, replace=False)

real_sample = real_images[idx_real]
fake_sample = fake_images[idx_fake]

# Plot
plt.figure(figsize=(7,6))

# -----------------------
# Real Images (Top Row)
# -----------------------
for i in range(8):
    plt.subplot(4, 4, i + 1)
    plt.imshow(real_sample[i].squeeze(), cmap='gray')
    plt.title("Real")
    plt.axis('off')

# -----------------------
# Fake Images (Bottom Row)
# -----------------------
for i in range(8):
    plt.subplot(4, 4, 8 + i + 1)
    plt.imshow(fake_sample[i].squeeze(), cmap='gray')
    plt.title("Fake")
    plt.axis('off')

plt.tight_layout()
plt.show()

# Version2

In [ ]:
# new
import matplotlib.pyplot as plt
import numpy as np

# Select 8 random images
idx_real = np.random.choice(len(real_images), 8, replace=False)
idx_fake = np.random.choice(len(fake_images), 8, replace=False)

real_sample = real_images[idx_real]
fake_sample = fake_images[idx_fake]

# Plot
plt.figure(figsize=(7,6))

# -----------------------
# Real Images (Top Row)
# -----------------------
for i in range(8):
    plt.subplot(4, 4, i + 1)
    plt.imshow(real_sample[i].squeeze(), cmap='gray')
    plt.title("Real")
    plt.axis('off')

# -----------------------
# Fake Images (Bottom Row)
# -----------------------
for i in range(8):
    plt.subplot(4, 4, 8 + i + 1)
    plt.imshow(fake_sample[i].squeeze(), cmap='gray')
    plt.title("Fake")
    plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# new
import matplotlib.pyplot as plt
import numpy as np

# Select 8 random images
idx_real = np.random.choice(len(real_images), 8, replace=False)
idx_fake = np.random.choice(len(fake_images), 8, replace=False)

real_sample = real_images[idx_real]
fake_sample = fake_images[idx_fake]

# Plot
plt.figure(figsize=(7,6))

# -----------------------
# Real Images (Top Row)
# -----------------------
for i in range(8):
    plt.subplot(4, 4, i + 1)
    plt.imshow(real_sample[i].squeeze(), cmap='gray')
    plt.title("Real")
    plt.axis('off')

# -----------------------
# Fake Images (Bottom Row)
# -----------------------
for i in range(8):
    plt.subplot(4, 4, 8 + i + 1)
    plt.imshow(fake_sample[i].squeeze(), cmap='gray')
    plt.title("Fake")
    plt.axis('off')

plt.tight_layout()
plt.show()

# Version3

In [ ]:
# new
import matplotlib.pyplot as plt
import numpy as np

# Select 8 random images
idx_real = np.random.choice(len(real_images), 8, replace=False)
idx_fake = np.random.choice(len(fake_images3), 8, replace=False)

real_sample = real_images[idx_real]
fake_sample = fake_images3[idx_fake]

# Plot
plt.figure(figsize=(7,6))

# -----------------------
# Real Images (Top Row)
# -----------------------
for i in range(8):
    plt.subplot(4, 4, i + 1)
    plt.imshow(real_sample[i].squeeze(), cmap='gray')
    plt.title("Real")
    plt.axis('off')

# -----------------------
# Fake Images (Bottom Row)
# -----------------------
for i in range(8):
    plt.subplot(4, 4, 8 + i + 1)
    plt.imshow(fake_sample[i].squeeze(), cmap='gray')
    plt.title("Fake")
    plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# new
import matplotlib.pyplot as plt
import numpy as np

# Select 8 random images
idx_real = np.random.choice(len(real_images), 8, replace=False)
idx_fake = np.random.choice(len(fake_images3), 8, replace=False)

real_sample = real_images[idx_real]
fake_sample = fake_images3[idx_fake]

# Plot
plt.figure(figsize=(7,6))

# -----------------------
# Real Images (Top Row)
# -----------------------
for i in range(8):
    plt.subplot(4, 4, i + 1)
    plt.imshow(real_sample[i].squeeze(), cmap='gray')
    plt.title("Real")
    plt.axis('off')

# -----------------------
# Fake Images (Bottom Row)
# -----------------------
for i in range(8):
    plt.subplot(4, 4, 8 + i + 1)
    plt.imshow(fake_sample[i].squeeze(), cmap='gray')
    plt.title("Fake")
    plt.axis('off')

plt.tight_layout()
plt.show()

# Version 1

In [ ]:
# last
import matplotlib.pyplot as plt
import numpy as np

# Select 20 random fake images
idx = np.random.choice(len(fake_images), 20, replace=False)
samples = fake_images[idx]

# Optional: convert [-1,1] → [0,1] for better visualization
def denormalize(img):
    return (img + 1) / 2

# Plot (4 rows × 5 columns)
plt.figure(figsize=(7,6))

for i in range(20):
    plt.subplot(4, 5, i + 1)
    plt.imshow(denormalize(samples[i]).squeeze(), cmap='gray')
    plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# last
import matplotlib.pyplot as plt
import numpy as np

# Select 20 random fake images
idx = np.random.choice(len(fake_images), 20, replace=False)
samples = fake_images[idx]

# Optional: convert [-1,1] → [0,1] for better visualization
def denormalize(img):
    return (img + 1) / 2

# Plot (4 rows × 5 columns)
plt.figure(figsize=(7,6))

for i in range(20):
    plt.subplot(4, 5, i + 1)
    plt.imshow(denormalize(samples[i]).squeeze(), cmap='gray')
    plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# last
import matplotlib.pyplot as plt
import numpy as np

# Select 20 random fake images
idx = np.random.choice(len(fake_images), 20, replace=False)
samples = fake_images[idx]

# Optional: convert [-1,1] → [0,1] for better visualization
def denormalize(img):
    return (img + 1) / 2

# Plot (4 rows × 5 columns)
plt.figure(figsize=(7,6))

for i in range(20):
    plt.subplot(4, 5, i + 1)
    plt.imshow(denormalize(samples[i]).squeeze(), cmap='gray')
    plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# last
import matplotlib.pyplot as plt
import numpy as np

# Select 20 random fake images
idx = np.random.choice(len(fake_images), 20, replace=False)
samples = fake_images[idx]

# Optional: convert [-1,1] → [0,1] for better visualization
def denormalize(img):
    return (img + 1) / 2

# Plot (4 rows × 5 columns)
plt.figure(figsize=(7,6))

for i in range(20):
    plt.subplot(4, 5, i + 1)
    plt.imshow(denormalize(samples[i]).squeeze(), cmap='gray')
    plt.axis('off')

plt.tight_layout()
plt.show()

# Version 2

In [ ]:
# new
import matplotlib.pyplot as plt
import numpy as np

# Select 20 random fake images
idx = np.random.choice(len(fake_images), 20, replace=False)
samples = fake_images[idx]

# Optional: convert [-1,1] → [0,1] for better visualization
def denormalize(img):
    return (img + 1) / 2

# Plot (4 rows × 5 columns)
plt.figure(figsize=(7,6))

for i in range(20):
    plt.subplot(4, 5, i + 1)
    plt.imshow(denormalize(samples[i]).squeeze(), cmap='gray')
    plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# new
import matplotlib.pyplot as plt
import numpy as np

# Select 20 random fake images
idx = np.random.choice(len(fake_images), 20, replace=False)
samples = fake_images[idx]

# Optional: convert [-1,1] → [0,1] for better visualization
def denormalize(img):
    return (img + 1) / 2

# Plot (4 rows × 5 columns)
plt.figure(figsize=(7,6))

for i in range(20):
    plt.subplot(4, 5, i + 1)
    plt.imshow(denormalize(samples[i]).squeeze(), cmap='gray')
    plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# new
import matplotlib.pyplot as plt
import numpy as np

# Select 20 random fake images
idx = np.random.choice(len(fake_images), 20, replace=False)
samples = fake_images[idx]

# Optional: convert [-1,1] → [0,1] for better visualization
def denormalize(img):
    return (img + 1) / 2

# Plot (4 rows × 5 columns)
plt.figure(figsize=(7,6))

for i in range(20):
    plt.subplot(4, 5, i + 1)
    plt.imshow(denormalize(samples[i]).squeeze(), cmap='gray')
    plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# new
import matplotlib.pyplot as plt
import numpy as np

# Select 20 random fake images
idx = np.random.choice(len(fake_images), 20, replace=False)
samples = fake_images[idx]

# Optional: convert [-1,1] → [0,1] for better visualization
def denormalize(img):
    return (img + 1) / 2

# Plot (4 rows × 5 columns)
plt.figure(figsize=(7,6))

for i in range(20):
    plt.subplot(4, 5, i + 1)
    plt.imshow(denormalize(samples[i]).squeeze(), cmap='gray')
    plt.axis('off')

plt.tight_layout()
plt.show()

# Version 3

In [ ]:
# new
import matplotlib.pyplot as plt
import numpy as np

# Select 20 random fake images
idx = np.random.choice(len(fake_images3), 20, replace=False)
samples = fake_images3[idx]

# Optional: convert [-1,1] → [0,1] for better visualization
def denormalize(img):
    return (img + 1) / 2

# Plot (4 rows × 5 columns)
plt.figure(figsize=(7,6))

for i in range(20):
    plt.subplot(4, 5, i + 1)
    plt.imshow(samples[i].squeeze(), cmap='gray')
    plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# new
import matplotlib.pyplot as plt
import numpy as np

# Select 20 random fake images
idx = np.random.choice(len(fake_images3), 20, replace=False)
samples = fake_images3[idx]

# Optional: convert [-1,1] → [0,1] for better visualization
def denormalize(img):
    return (img + 1) / 2

# Plot (4 rows × 5 columns)
plt.figure(figsize=(7,6))

for i in range(20):
    plt.subplot(4, 5, i + 1)
    plt.imshow(samples[i].squeeze(), cmap='gray')
    plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# new
import matplotlib.pyplot as plt
import numpy as np

# Select 20 random fake images
idx = np.random.choice(len(fake_images3), 20, replace=False)
samples = fake_images3[idx]

# Optional: convert [-1,1] → [0,1] for better visualization
def denormalize(img):
    return (img + 1) / 2

# Plot (4 rows × 5 columns)
plt.figure(figsize=(7,6))

for i in range(20):
    plt.subplot(4, 5, i + 1)
    plt.imshow(samples[i].squeeze(), cmap='gray')
    plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# new
import matplotlib.pyplot as plt
import numpy as np

# Select 20 random fake images
idx = np.random.choice(len(fake_images3), 20, replace=False)
samples = fake_images3[idx]

# Optional: convert [-1,1] → [0,1] for better visualization
def denormalize(img):
    return (img + 1) / 2

# Plot (4 rows × 5 columns)
plt.figure(figsize=(7,6))

for i in range(20):
    plt.subplot(4, 5, i + 1)
    plt.imshow(samples[i].squeeze(), cmap='gray')
    plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# new
import matplotlib.pyplot as plt
import numpy as np

# Select 20 random fake images
idx = np.random.choice(len(fake_images3), 20, replace=False)
samples = fake_images3[idx]

# Optional: convert [-1,1] → [0,1] for better visualization
def denormalize(img):
    return (img + 1) / 2

# Plot (4 rows × 5 columns)
plt.figure(figsize=(7,6))

for i in range(20):
    plt.subplot(4, 5, i + 1)
    plt.imshow(samples[i].squeeze(), cmap='gray')
    plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# new
import matplotlib.pyplot as plt
import numpy as np

# Select 20 random fake images
idx = np.random.choice(len(fake_images3), 20, replace=False)
samples = fake_images3[idx]

# Optional: convert [-1,1] → [0,1] for better visualization
def denormalize(img):
    return (img + 1) / 2

# Plot (4 rows × 5 columns)
plt.figure(figsize=(7,6))

for i in range(20):
    plt.subplot(4, 5, i + 1)
    plt.imshow(samples[i].squeeze(), cmap='gray')
    plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# new
import matplotlib.pyplot as plt
import numpy as np

# Select 20 random fake images
idx = np.random.choice(len(fake_images3), 20, replace=False)
samples = fake_images3[idx]

# Optional: convert [-1,1] → [0,1] for better visualization
def denormalize(img):
    return (img + 1) / 2

# Plot (4 rows × 5 columns)
plt.figure(figsize=(7,6))

for i in range(20):
    plt.subplot(4, 5, i + 1)
    plt.imshow(samples[i].squeeze(), cmap='gray')
    plt.axis('off')

plt.tight_layout()
plt.show()

# Additional Calculation. Not needed

In [ ]:
# Shuffle both
real_idx = np.random.choice(len(real_images), 16, replace=False)
fake_idx = np.random.choice(len(fake_images), 16, replace=False)

real_sample = real_images[real_idx]
fake_sample = fake_images[fake_idx]

plt.figure(figsize=(16,8))  # 🔥 bigger

# 🔵 Real images (TOP ROWS)
for i in range(16):
    plt.subplot(4,8,i+1)
    plt.imshow(real_sample[i,:,:,0], cmap='gray')
    plt.axis('off')

# 🟢 Fake images (BOTTOM ROWS)
for i in range(16):
    plt.subplot(4,8,i+17)
    plt.imshow(fake_sample[i,:,:,0], cmap='gray')
    plt.axis('off')

plt.suptitle("Top: Shuffled Real MRI | Bottom: Shuffled Generated MRI (DCGAN)", fontsize=20)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12,12))

for i in range(8):
    # Real
    plt.subplot(8,2,2*i+1)
    plt.imshow(real_sample[i,:,:,0], cmap='gray')
    plt.title("Real", fontsize=12)
    plt.axis('off')

    # Fake
    plt.subplot(8,2,2*i+2)
    plt.imshow(fake_sample[i,:,:,0], cmap='gray')
    plt.title("Generated", fontsize=12)
    plt.axis('off')

plt.suptitle("Real vs Generated MRI (Shuffled)", fontsize=18)
plt.tight_layout()
plt.show()

In [ ]:
!pip install torchmetrics

In [ ]:
!pip install torchmetrics[image] torch-fidelity

In [ ]:
import numpy as np

# Assuming generated_images is your GAN output (128x128x1)
np.save("generated_images_128.npy", generated_images)
print("Generated images saved successfully!")

In [ ]:
import numpy as np

generated_images = np.load("generated_images_128.npy")
print("Shape:", generated_images.shape)

# do not need to run from there

In [ ]:
import tensorflow as tf
import numpy as np

# Number of images to evaluate
NUM_EVAL = 5000

# Real images (shuffle)
real_idx = np.random.choice(len(tumor_images), NUM_EVAL, replace=False)
real_eval = tumor_images[real_idx]
real_eval = (real_eval + 1) / 2  # scale to [0,1]

# Fake images (already generated)
fake_eval = generated_images[:NUM_EVAL]

In [ ]:
!pip install tensorflow-gan

In [ ]:
import tensorflow_gan as tfgan

def preprocess_for_fid(images):
    # Resize to Inception size and convert to 3 channels
    images = tf.image.resize(images, (299,299))
    images = tf.image.grayscale_to_rgb(images)
    return images

BATCH_SIZE = 64  # small batch to avoid GPU OOM

real_batches = []
fake_batches = []

for i in range(0, NUM_EVAL, BATCH_SIZE):
    real_batch = preprocess_for_fid(real_eval[i:i+BATCH_SIZE])
    fake_batch = preprocess_for_fid(fake_eval[i:i+BATCH_SIZE])

    real_batches.append(real_batch)
    fake_batches.append(fake_batch)

real_batches = tf.concat(real_batches, axis=0)
fake_batches = tf.concat(fake_batches, axis=0)

fid_score = tfgan.eval.frechet_inception_distance(real_batches, fake_batches)
print("FID Score:", fid_score.numpy())

In [ ]:
import torch
from torchmetrics.image.fid import FrechetInceptionDistance
import torch.nn.functional as F

# Convert to torch tensors
real_tensor = torch.tensor((real_eval*255).astype(np.uint8)).repeat(1,3,1,1).float()  # 3 channels
fake_tensor = torch.tensor((fake_eval*255).astype(np.uint8)).repeat(1,3,1,1).float()  # 3 channels

# FID metric
fid = FrechetInceptionDistance(feature=2048, reset_real_features=True)

BATCH = 32
for i in range(0, NUM_EVAL, BATCH):
    fid.update(real_tensor[i:i+BATCH], real=True)
    fid.update(fake_tensor[i:i+BATCH], real=False)

fid_score = fid.compute()
print("FID Score:", fid_score.item())

In [ ]:
import torch
from torchmetrics.image.fid import FrechetInceptionDistance
import numpy as np

# Number of images to evaluate
NUM_EVAL = 5000

# Real images (scaled [0,1])
real_eval = (tumor_images[:NUM_EVAL] + 1)/2
fake_eval = generated_images[:NUM_EVAL]

# Convert to torch tensors with 3 channels
real_tensor = torch.tensor((real_eval*255).astype(np.uint8)).repeat(1,3,1,1)
fake_tensor = torch.tensor((fake_eval*255).astype(np.uint8)).repeat(1,3,1,1)

In [ ]:
fid = FrechetInceptionDistance(feature=2048, reset_real_features=True)

BATCH = 32

for i in range(0, NUM_EVAL, BATCH):
    fid.update(real_tensor[i:i+BATCH], real=True)
    fid.update(fake_tensor[i:i+BATCH], real=False)

fid_score = fid.compute()
print("FID Score:", fid_score.item())

In [ ]:
# tumor_images is still in memory, or you can load a smaller subset if needed
NUM_EVAL = 2000  # pick a subset to save memory
indices = np.random.choice(len(tumor_images), NUM_EVAL, replace=False)
real_eval = tumor_images[indices]
fake_eval = generated_images[:NUM_EVAL]  # pick the same number

In [ ]:
import os
import nibabel as nib
import cv2
import numpy as np
from tqdm import tqdm

IMG_SIZE = 128
extract_path = "/kaggle/working/brats2021"
NUM_EVAL = 2000  # number of real images to use

real_images = []

patient_folders = sorted([
    f for f in os.listdir(extract_path)
    if os.path.isdir(os.path.join(extract_path, f)) and "BraTS" in f
])

print("Total patients:", len(patient_folders))

# Loop until we have NUM_EVAL images
while len(real_images) < NUM_EVAL:
    for patient in tqdm(patient_folders, desc="Loading real images"):
        patient_path = os.path.join(extract_path, patient)
        files = os.listdir(patient_path)

        flair_path = [f for f in files if "flair" in f][0]
        seg_path   = [f for f in files if "seg" in f][0]

        flair = nib.load(os.path.join(patient_path, flair_path)).get_fdata()
        seg   = nib.load(os.path.join(patient_path, seg_path)).get_fdata()

        start = int(flair.shape[2]*0.2)
        end   = int(flair.shape[2]*0.8)

        for i in range(start, end):
            tumor_pixels = np.sum(seg[:,:,i]>0)
            brain_pixels = np.sum(flair[:,:,i]>0)

            if tumor_pixels>200 and brain_pixels>15000:
                img_slice = flair[:,:,i]
                coords = np.argwhere(img_slice > np.percentile(img_slice,5))
                if coords.shape[0]==0:
                    continue

                y_min, x_min = coords.min(axis=0)
                y_max, x_max = coords.max(axis=0)

                pad = 10
                y_min = max(0,y_min-pad)
                x_min = max(0,x_min-pad)
                y_max = min(img_slice.shape[0],y_max+pad)
                x_max = min(img_slice.shape[1],x_max+pad)

                cropped_img = img_slice[y_min:y_max, x_min:x_max]
                img = cv2.resize(cropped_img, (IMG_SIZE, IMG_SIZE))
                img = (img - np.mean(img)) / (np.std(img)+1e-8)   # normalize
                img = (img - img.min()) / (img.max()-img.min()+1e-8)
                img = img*2-1
                img = np.expand_dims(img, axis=-1)

                real_images.append(img)

                if len(real_images) >= NUM_EVAL:
                    break
        if len(real_images) >= NUM_EVAL:
            break

real_images = np.array(real_images, dtype=np.float32)
print("Loaded real images shape:", real_images.shape)

In [ ]:
# generated_images is already saved as generated_images_128.npy
import numpy as np
generated_images = np.load("generated_images_128.npy")

# Shuffle both real and fake images
np.random.shuffle(real_images)
np.random.shuffle(generated_images)

# Pick same number
NUM_EVAL = min(len(real_images), len(generated_images))
real_eval = real_images[:NUM_EVAL]
fake_eval = generated_images[:NUM_EVAL]

In [ ]:
# -------------------------------
# Shuffle and Match Sizes
# -------------------------------
np.random.shuffle(real_images)
np.random.shuffle(generated_images)
NUM_EVAL = min(len(real_images), len(generated_images))
real_eval = real_images[:NUM_EVAL]
fake_eval = generated_images[:NUM_EVAL]


# Start from here

In [ ]:
import os
import numpy as np
import nibabel as nib
import cv2
import tensorflow as tf
from tqdm import tqdm
import matplotlib.pyplot as plt
from skimage.metrics import structural_similarity as ssim
from scipy.linalg import sqrtm
import torch
from torchmetrics.image.fid import FrechetInceptionDistance

In [ ]:
# -------------------------------
# Settings
# -------------------------------
IMG_SIZE = 128
extract_path = "/kaggle/working/brats2021"
NUM_EVAL = 2000  # Number of real images to sample for evaluation
BATCH = 32       # For batch-safe FID


In [ ]:
# -------------------------------
# Load Generated Images
# -------------------------------
generated_images = np.load("generated_images_128.npy")
print("Generated images loaded:", generated_images.shape)


In [ ]:
# -------------------------------
# Load Real Images (subset)
# -------------------------------
real_images = []
patient_folders = sorted([
    f for f in os.listdir(extract_path)
    if os.path.isdir(os.path.join(extract_path, f)) and "BraTS" in f
])

while len(real_images) < NUM_EVAL:
    for patient in tqdm(patient_folders, desc="Loading real images"):
        patient_path = os.path.join(extract_path, patient)
        files = os.listdir(patient_path)
        flair_path = [f for f in files if "flair" in f][0]
        seg_path   = [f for f in files if "seg" in f][0]

        flair = nib.load(os.path.join(patient_path, flair_path)).get_fdata()
        seg   = nib.load(os.path.join(patient_path, seg_path)).get_fdata()

        start = int(flair.shape[2]*0.2)
        end   = int(flair.shape[2]*0.8)
        for i in range(start, end):
            tumor_pixels = np.sum(seg[:,:,i]>0)
            brain_pixels = np.sum(flair[:,:,i]>0)
            if tumor_pixels>200 and brain_pixels>15000:
                img_slice = flair[:,:,i]
                coords = np.argwhere(img_slice > np.percentile(img_slice,5))
                if coords.shape[0]==0:
                    continue

                y_min, x_min = coords.min(axis=0)
                y_max, x_max = coords.max(axis=0)
                pad = 10
                y_min = max(0,y_min-pad)
                x_min = max(0,x_min-pad)
                y_max = min(img_slice.shape[0],y_max+pad)
                x_max = min(img_slice.shape[1],x_max+pad)

                cropped_img = img_slice[y_min:y_max, x_min:x_max]
                img = cv2.resize(cropped_img, (IMG_SIZE, IMG_SIZE))
                # Normalize to [-1,1]
                img = (img - np.mean(img))/(np.std(img)+1e-8)
                img = (img - img.min())/(img.max()-img.min()+1e-8)
                img = img*2 - 1
                img = np.expand_dims(img, axis=-1)
                real_images.append(img)
                if len(real_images) >= NUM_EVAL:
                    break
        if len(real_images) >= NUM_EVAL:
            break

real_images = np.array(real_images, dtype=np.float32)
print("Loaded real images shape:", real_images.shape)


In [ ]:
# -------------------------------
# Shuffle and Match Sizes
# -------------------------------
np.random.shuffle(real_images)
np.random.shuffle(generated_images)
NUM_EVAL = min(len(real_images), len(generated_images))
real_eval = real_images[:NUM_EVAL]
fake_eval = generated_images[:NUM_EVAL]

In [ ]:
import numpy as np
import torch
from torchmetrics.image.fid import FrechetInceptionDistance
from skimage.metrics import structural_similarity as ssim

# Assume real_eval and fake_eval are [N, 128,128,1] floats in [-1,1]

# Convert to [0,255] uint8 and 3 channels
def to_3ch_tensor(imgs):
    # [-1,1] -> [0,255]
    imgs = ((imgs+1)/2*255).astype(np.uint8)
    # [N,H,W,1] -> [N,3,H,W]
    imgs = np.repeat(imgs, 3, axis=-1)
    # Convert to torch tensor and permute
    imgs = torch.from_numpy(imgs).permute(0,3,1,2)  # [N,3,H,W]
    return imgs

real_tensor = to_3ch_tensor(real_eval)
fake_tensor = to_3ch_tensor(fake_eval)

# Initialize FID metric
fid = FrechetInceptionDistance(feature=2048, reset_real_features=True)

BATCH = 32
NUM_EVAL = len(real_tensor)

for i in range(0, NUM_EVAL, BATCH):
    fid.update(real_tensor[i:i+BATCH], real=True)
    fid.update(fake_tensor[i:i+BATCH], real=False)

fid_score = fid.compute()
print("FID Score:", fid_score.item())

# Compute average SSIM (grayscale)
ssim_scores = [ssim(real_eval[i,:,:,0], fake_eval[i,:,:,0], data_range=2.0) for i in range(NUM_EVAL)]
avg_ssim = np.mean(ssim_scores)
print("Average SSIM:", avg_ssim)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# -------------------------------
# Settings
# -------------------------------
NUM_DISPLAY = 16  # number of images to show
indices_real = np.random.choice(len(real_eval), NUM_DISPLAY, replace=False)
indices_fake = np.random.choice(len(fake_eval), NUM_DISPLAY, replace=False)

real_samples = real_eval[indices_real]
fake_samples = fake_eval[indices_fake]

# -------------------------------
# Plot
# -------------------------------
plt.figure(figsize=(12,6))

for i in range(NUM_DISPLAY):
    # Real
    plt.subplot(2, NUM_DISPLAY, i+1)
    plt.imshow(real_samples[i,:,:,0], cmap='gray')
    plt.axis('off')
    if i == NUM_DISPLAY // 2:
        plt.title("Real Images", fontsize=14)

    # Fake
    plt.subplot(2, NUM_DISPLAY, i+1+NUM_DISPLAY)
    plt.imshow(fake_samples[i,:,:,0], cmap='gray')
    plt.axis('off')
    if i == NUM_DISPLAY // 2:
        plt.title("Generated Images", fontsize=14)

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# -------------------------------
# Settings
# -------------------------------
NUM_DISPLAY = 32  # total number of images to display
cols = 8          # number of images per row
rows = NUM_DISPLAY // cols * 2  # double for real + fake rows

# Shuffle and pick samples
indices_real = np.random.choice(len(real_eval), NUM_DISPLAY, replace=False)
indices_fake = np.random.choice(len(fake_eval), NUM_DISPLAY, replace=False)
real_samples = real_eval[indices_real]
fake_samples = fake_eval[indices_fake]

# -------------------------------
# Plot
# -------------------------------
plt.figure(figsize=(24, 12))  # increase size for clarity

for i in range(NUM_DISPLAY):
    # Real image
    plt.subplot(rows, cols, i+1)
    plt.imshow(real_samples[i,:,:,0], cmap='gray')
    plt.axis('off')
    if i == cols // 2:
        plt.title("Real Images", fontsize=16)

    # Fake image
    plt.subplot(rows, cols, i+1+NUM_DISPLAY)
    plt.imshow(fake_samples[i,:,:,0], cmap='gray')
    plt.axis('off')
    if i == cols // 2:
        plt.title("Generated Images", fontsize=16)

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# -------------------------------
# Settings
# -------------------------------
NUM_DISPLAY = 16  # total images to show (real + fake)
cols = 4          # number of images per row
rows = 2          # real and fake only 1 row each
fig_width = 20    # increase figure width
fig_height = 10   # increase figure height

# Shuffle and pick samples
indices_real = np.random.choice(len(real_eval), NUM_DISPLAY, replace=False)
indices_fake = np.random.choice(len(fake_eval), NUM_DISPLAY, replace=False)
real_samples = real_eval[indices_real]
fake_samples = fake_eval[indices_fake]

# -------------------------------
# Plot
# -------------------------------
plt.figure(figsize=(fig_width, fig_height))

for i in range(NUM_DISPLAY):
    # Real image
    plt.subplot(rows, NUM_DISPLAY, i+1)
    plt.imshow(real_samples[i,:,:,0], cmap='gray')
    plt.axis('off')
    if i == NUM_DISPLAY // 2:
        plt.title("Real Images", fontsize=16)

    # Fake image
    plt.subplot(rows, NUM_DISPLAY, i+1+NUM_DISPLAY)
    plt.imshow(fake_samples[i,:,:,0], cmap='gray')
    plt.axis('off')
    if i == NUM_DISPLAY // 2:
        plt.title("Generated Images", fontsize=16)

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# -------------------------------
# Settings
# -------------------------------
NUM_DISPLAY = 8   # 8 images each for real and fake
cols = 4          # number of images per row
rows = 2          # number of rows per set
fig_width = 6   # figure width
fig_height = 4    # figure height

# Shuffle and pick samples
indices_real = np.random.choice(len(real_eval), NUM_DISPLAY, replace=False)
indices_fake = np.random.choice(len(fake_eval), NUM_DISPLAY, replace=False)
real_samples = real_eval[indices_real]
fake_samples = fake_eval[indices_fake]

# -------------------------------
# Plot Real Images
# -------------------------------
plt.figure(figsize=(fig_width, fig_height))
for i in range(NUM_DISPLAY):
    plt.subplot(rows, cols, i + 1)
    plt.imshow(real_samples[i, :, :, 0], cmap='gray')
    plt.axis('off')
plt.suptitle("Real Images", fontsize=18)
plt.tight_layout()
plt.show()

# -------------------------------
# Plot Fake Images
# -------------------------------
plt.figure(figsize=(fig_width, fig_height))
for i in range(NUM_DISPLAY):
    plt.subplot(rows, cols, i + 1)
    plt.imshow(fake_samples[i, :, :, 0], cmap='gray')
    plt.axis('off')
plt.suptitle("Generated Images", fontsize=18)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# -------------------------------
# Settings
# -------------------------------
NUM_DISPLAY = 20   # number of fake images to display
cols = 5           # number of images per row
rows = 4           # number of rows
fig_width = 8
fig_height = 6

# Shuffle and pick samples
indices_fake = np.random.choice(len(fake_eval), NUM_DISPLAY, replace=False)
fake_samples = fake_eval[indices_fake]

# -------------------------------
# Plot Fake Images
# -------------------------------
plt.figure(figsize=(fig_width, fig_height))
for i in range(NUM_DISPLAY):
    plt.subplot(rows, cols, i + 1)
    plt.imshow(fake_samples[i, :, :, 0], cmap='gray')
    plt.axis('off')

plt.suptitle("Generated Images", fontsize=18)
plt.tight_layout()
plt.show()

In [ ]:
import torch
import torch.nn.functional as F
from torchvision.models import inception_v3
from torchvision import transforms
import numpy as np

# 1. Load pretrained InceptionV3
inception = inception_v3(pretrained=True, transform_input=False).eval()
if torch.cuda.is_available():
    inception = inception.cuda()

# 2. Preprocessing: convert grayscale to 3-channel RGB
preprocess = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((299, 299)),
    transforms.Lambda(lambda img: img.convert("RGB")),  # Convert 1-channel to 3-channel
    transforms.ToTensor(),
])

def inception_score(images, batch_size=10, splits=1):
    N = len(images)
    preds = []

    for i in range(0, N, batch_size):
        batch = images[i:i+batch_size]
        batch_tensor = torch.stack([preprocess(img) for img in batch])
        if torch.cuda.is_available():
            batch_tensor = batch_tensor.cuda()
        with torch.no_grad():
            pred = inception(batch_tensor)
            pred = F.softmax(pred, dim=1).cpu().numpy()
        preds.append(pred)

    preds = np.vstack(preds)
    # Marginal distribution
    p_y = np.mean(preds, axis=0)

    # KL divergence for each image
    kl_divs = preds * (np.log(preds + 1e-16) - np.log(p_y + 1e-16))
    kl_divs = np.sum(kl_divs, axis=1)
    score = np.exp(np.mean(kl_divs))
    return score

# Example usage:
images = fake_eval  # your fake images
print("Inception Score:", inception_score(images))